# 01 - 飞书认证 & Token 获取测试

测试内容:
1. 从环境变量读取 App ID / App Secret
2. 获取 tenant_access_token
3. Token 缓存机制
4. 健康检查

参考文档: https://open.feishu.cn/document/server-docs/authentication-management/access-token/tenant_access_token_internal

In [1]:
import os
import sys
import json
from pathlib import Path

# 加载 .env 文件（如果存在）
try:
    from dotenv import load_dotenv
    env_path = Path(__file__).parent.parent.parent / '.env' if '__file__' in dir() else Path('../../.env')
    if env_path.exists():
        load_dotenv(env_path)
        print(f"✓ 已加载 .env: {env_path.absolute()}")
except ImportError:
    pass

# 检查环境变量
app_id = os.environ.get('FEISHU_APP_ID') or os.environ.get('LARK_APP_ID')
app_secret = os.environ.get('FEISHU_APP_SECRET') or os.environ.get('LARK_APP_SECRET')

print(f"FEISHU_APP_ID / LARK_APP_ID: {'✓ 已设置' if app_id else '✗ 未设置'}")
print(f"FEISHU_APP_SECRET / LARK_APP_SECRET: {'✓ 已设置' if app_secret else '✗ 未设置'}")

if not app_id:
    print("\n⚠ 请设置环境变量:")
    print("  FEISHU_APP_ID=cli_xxx")
    print("  FEISHU_APP_SECRET=xxx")
    print("\n或在当前目录创建 .env 文件")

✓ 已加载 .env: d:\ALL IN AI\MetaBlog\docs-internal\feishu-api-lab\..\..\.env
FEISHU_APP_ID / LARK_APP_ID: ✓ 已设置
FEISHU_APP_SECRET / LARK_APP_SECRET: ✓ 已设置


In [2]:
from feishu_client import FeishuClient

# 初始化客户端
client = FeishuClient()

# 健康检查
health = client.health_check()
print(json.dumps(health, indent=2, ensure_ascii=False))

{
  "ok": true,
  "token_valid": true,
  "expire_at": 1776531165.9161603,
  "expire_in": 6193
}


In [3]:
# 手动获取 token
token = client.get_tenant_access_token()
print(f"Token 长度: {len(token)}")
print(f"Token 前缀: {token[:20]}...")
print(f"过期时间: {client._expire_at}")
print(f"剩余有效期: {int(client._expire_at - __import__('time').time())} 秒")

Token 长度: 42
Token 前缀: t-g1044imQMPWJ5KFYKL...
过期时间: 1776531165.9161603
剩余有效期: 6193 秒


In [4]:
# 测试 Token 缓存：再次获取应该直接返回缓存的 token
import time

start = time.time()
token2 = client.get_tenant_access_token()
elapsed = time.time() - start

print(f"第二次获取耗时: {elapsed*1000:.1f} ms (缓存命中应该 < 1ms)")
print(f"两次 token 相同: {token == token2}")

第二次获取耗时: 0.0 ms (缓存命中应该 < 1ms)
两次 token 相同: True


In [5]:
# 测试 force_refresh
start = time.time()
token3 = client.get_tenant_access_token(force_refresh=True)
elapsed = time.time() - start

print(f"强制刷新耗时: {elapsed*1000:.1f} ms")
print(f"新 token 与旧 token 相同: {token == token3}")

强制刷新耗时: 168.4 ms
新 token 与旧 token 相同: True


In [6]:
# 查看缓存文件
print(f"缓存文件路径: {client._cache_path}")
if client._cache_path.exists():
    cache_data = json.loads(client._cache_path.read_text())
    print(f"缓存内容: {json.dumps({k: v if k != 'token' else v[:20]+'...' for k, v in cache_data.items()}, indent=2)}")
else:
    print("缓存文件不存在")

缓存文件路径: C:\Users\ADMINI~1\AppData\Local\Temp\.feishu_token_70f8dbd6.json
缓存内容: {
  "token": "t-g1044imQMPWJ5KFYKL...",
  "expire_at": 1776531165.8900201
}


In [7]:
# 清理缓存
client.clear_cache()
print("✓ Token 缓存已清除")

✓ Token 缓存已清除
